# 02 – Vet clinics cleaning and normalization (v1)

This notebook takes the v0 dataset (OSM + LOR context) and produces a cleaned
and normalized vet clinics table (v1), plus a reduced export table for the
unified POI layer.

Goals:

- Preserve OSM numeric `id` as the primary identifier.
- Keep `geometry` for spatial joins and unified POI.
- Normalise address, contact, and operational fields.
- Use NULLs (`NaN` / `pd.NA`) for missing data, not empty strings.
- Optionally use Nominatim to backfill missing addresses.
- Provide a reduced export schema for app/backend usage.


In [1]:
from pathlib import Path
import os

ROOT = Path.cwd()
if (ROOT / "veterinary_clinics" / "sources").exists():
    # If we are at repo root, move into veterinary_clinics
    os.chdir(ROOT / "veterinary_clinics")
    print("Changed CWD to:", Path.cwd())
else:
    # If the notebook is already inside veterinary_clinics, keep as is
    print("Current CWD:", ROOT)
    print("Assuming this notebook already runs inside `veterinary_clinics`.")

import pandas as pd
import numpy as np

V0_PATH = Path("cache/vets_osm_berlin_with_lor_latest_v0.csv")

# IMPORTANT: keep IDs as strings so we do not lose leading zeros
df = pd.read_csv(
    V0_PATH,
    dtype={
        "district_id": "string",
        "neighborhood_id": "string",
        "lor_id": "string",
    },
)

print("v0 shape:", df.shape)
df.head()

Current CWD: /Users/jorge/Projects/layered-populate-data-pool-da/veterinary_clinics
Assuming this notebook already runs inside `veterinary_clinics`.
v0 shape: (175, 27)


,id,element,source_osm_id,name,addr:street,addr:housenumber,addr:postcode,addr:city,phone,contact:phone,...,wheelchair:description,emergency,lat,lon,geometry,lor_id,district,district_id,neighborhood,neighborhood_id
0,268917040,node,node/268917040,Tierarztpraxis am Urban,Baerwaldstraße,69,10961.0,Berlin,NaN,NaN,...,NaN,NaN,52.495684,13.405233,POINT (13.4052329 52.4956842),re_ortsteil.0202,Friedrichshain-Kreuzberg,11002002,Kreuzberg,0202
1,299795048,node,node/299795048,Dr. med. vet. Elke Hartwig,Straße 48,67,13125.0,Berlin,+49 30 9437820,NaN,...,NaN,NaN,52.606286,13.479555,POINT (13.4795548 52.60628629999999),re_ortsteil.0305,Pankow,11003003,Karow,0305
2,347294456,node,node/347294456,Tierarztpraxis Dr. Bernhard Sörensen,Königsberger Straße,36,12207.0,Berlin,+49 30 7738321,NaN,...,NaN,NaN,52.429722,13.320133,POINT (13.3201326 52.4297216),re_ortsteil.0602,Steglitz-Zehlendorf,11006006,Lichterfelde,0602
3,394867279,node,node/394867279,Tierarztpraxis Jeanette Koepsel,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,52.535199,13.270573,POINT (13.2705734 52.5351995),re_ortsteil.0503,Spandau,11005005,Siemensstadt,0503
4,411550894,node,node/411550894,Kleintierarztpraxis Berlin Kaulsdorf,Planitzstraße,19,12621.0,Berlin,+49 30 53018585,NaN,...,NaN,NaN,52.509511,13.589635,POINT (13.5896353 52.50951139999999),re_ortsteil.1003,Marzahn-Hellersdorf,11010010,Kaulsdorf,1003


## 1. Quick inspection of v0 dataset

We briefly check available columns to understand which raw attributes
we can map into the target schema.

In [2]:
df.columns.tolist()

['id',
 'element',
 'source_osm_id',
 'name',
 'addr:street',
 'addr:housenumber',
 'addr:postcode',
 'addr:city',
 'phone',
 'contact:phone',
 'email',
 'contact:email',
 'website',
 'contact:website',
 'opening_hours',
 'operator',
 'wheelchair',
 'wheelchair:description',
 'emergency',
 'lat',
 'lon',
 'geometry',
 'lor_id',
 'district',
 'district_id',
 'neighborhood',
 'neighborhood_id']

## 2. Helper functions

Utility functions to standardise string handling, build full addresses
and derive simple operating-day labels from the raw `opening_hours` field.

In [3]:
# Helper to convert any series to pandas string dtype with trimmed values,
# preserving missing values as <NA>
def to_str(series: pd.Series) -> pd.Series:
    s = series.astype("string")
    return s.str.strip()

def build_full_address(row) -> str | None:
    """Build a human-readable full address from street, number, postcode and city."""
    parts = []

    if pd.notna(row.get("addr_street")):
        street = row["addr_street"]
        if pd.notna(row.get("addr_housenumber")):
            street = f"{street} {row['addr_housenumber']}"
        parts.append(street)

    if pd.notna(row.get("addr_postcode")) or pd.notna(row.get("addr_city")):
        city_part = " ".join(
            str(x)
            for x in [row.get("addr_postcode"), row.get("addr_city")]
            if pd.notna(x)
        ).strip()
        if city_part:
            parts.append(city_part)

    if not parts:
        return None

    return ", ".join(parts)

def infer_operating_days(opening_hours: str | None) -> str | None:
    """
    Very simple heuristic to summarise opening hours into a high-level
    'operating_days' label (Mon–Fri / Mon–Sat / Mon–Sun).
    """
    if pd.isna(opening_hours):
        return None

    s = opening_hours.lower()
    if "su" in s or "sun" in s:
        return "Mon–Sun"
    if "sa" in s or "sat" in s:
        return "Mon–Sat"
    if any(d in s for d in ["mo", "tu", "we", "th", "fr"]):
        return "Mon–Fri"

    return None

## 3. Build normalized vet clinics table (`df_clean`)

Here we construct the cleaned table from the v0 dataset. We:

- Keep the numeric OSM `id` as the primary identifier.
- Standardise clinic name, address components, district / neighborhood fields.
- Preserve `geometry`, `latitude`, and `longitude`.
- Derive `services_offered`, operating metadata, and contact fields.

In [4]:
df_clean = pd.DataFrame()

# 3.1 Identifier (numeric OSM id as string)
if "id" in df.columns:
    df_clean["id"] = df["id"].astype("Int64").astype("string")
elif "source_osm_id" in df.columns:
    # Extract numeric part from e.g. "node/268917040" or "way/12345"
    df_clean["id"] = (
        df["source_osm_id"]
        .astype("string")
        .str.extract(r"(\d+)$", expand=False)
    )
else:
    # Fallback: use row index if nothing else is available
    df_clean["id"] = df.index.astype(str)

# 3.2 Clinic name (fallback chain: name -> operator -> address)
name_raw = to_str(df["name"])
operator_raw = to_str(df["operator"])

addr_street_raw = to_str(df["addr:street"])
addr_housenumber_raw = to_str(df["addr:housenumber"])

fallback_name = addr_street_raw.copy()
has_hn = addr_housenumber_raw.notna()
fallback_name = fallback_name.where(~has_hn, fallback_name + " " + addr_housenumber_raw)

clinic_name = name_raw.copy()
clinic_name = clinic_name.fillna(operator_raw)
clinic_name = clinic_name.fillna(fallback_name)

df_clean["clinic_name"] = clinic_name

# 3.3 Address components (raw)
df_clean["addr_street"] = addr_street_raw
df_clean["addr_housenumber"] = addr_housenumber_raw
df_clean["addr_postcode"] = to_str(df["addr:postcode"])
df_clean["addr_city"] = to_str(df["addr:city"])

# 3.4 District / neighborhood / IDs
df_clean["district"] = to_str(df["district"])
df_clean["district_id"] = to_str(df["district_id"])
df_clean["neighborhood"] = to_str(df["neighborhood"])
df_clean["neighborhood_id"] = to_str(df["neighborhood_id"]).str.zfill(4)
df_clean["lor_id"] = to_str(df["lor_id"])

# 3.5 Coordinates and geometry
df_clean["latitude"] = df["lat"]
df_clean["longitude"] = df["lon"]
df_clean["geometry"] = df["geometry"]

# 3.6 Services and operating hours
if "emergency" in df.columns:
    emergency_raw = to_str(df["emergency"])
else:
    emergency_raw = pd.Series(pd.NA, index=df.index, dtype="string")

# Initialise services_offered with all NULLs
services = pd.Series(pd.NA, index=df.index, dtype="string")

# True where emergency == "yes" (case-insensitive). <NA> values are treated as False.
mask_emergency = emergency_raw.str.lower().eq("yes")
services.loc[mask_emergency.fillna(False)] = "emergency"

df_clean["services_offered"] = services

opening_hours_raw = to_str(df["opening_hours"])
df_clean["operating_hours"] = opening_hours_raw
df_clean["operating_days"] = opening_hours_raw.map(infer_operating_days)

# 3.7 Contact details (raw + main fields)
phone_raw = to_str(df.get("phone"))
contact_phone_raw = to_str(df.get("contact:phone"))
email_raw = to_str(df.get("email"))
contact_email_raw = to_str(df.get("contact:email"))
website_raw = to_str(df.get("website"))
contact_website_raw = to_str(df.get("contact:website"))

df_clean["phone_raw"] = phone_raw
df_clean["contact_phone_raw"] = contact_phone_raw
df_clean["email_raw"] = email_raw
df_clean["contact_email_raw"] = contact_email_raw
df_clean["website_raw"] = website_raw
df_clean["contact_website_raw"] = contact_website_raw

df_clean["phone_main"] = phone_raw.fillna(contact_phone_raw)
df_clean["email_main"] = email_raw.fillna(contact_email_raw)
df_clean["website_main"] = website_raw.fillna(contact_website_raw)

# Combined contact_info (for display and fuzzy search)
def build_contact_info(row) -> str | None:
    parts = []
    if pd.notna(row["phone_main"]):
        parts.append(f"phone: {row['phone_main']}")
    if pd.notna(row["email_main"]):
        parts.append(f"email: {row['email_main']}")
    if pd.notna(row["website_main"]):
        parts.append(f"web: {row['website_main']}")
    return "; ".join(parts) if parts else None

df_clean["contact_info"] = (
    df_clean.apply(build_contact_info, axis=1)
    .astype("string")
)

# 3.8 Accessibility info
wheelchair_raw = to_str(df.get("wheelchair"))
wheelchair_desc = to_str(df.get("wheelchair:description"))

accessibility = wheelchair_raw.copy()
has_desc = wheelchair_desc.notna()
accessibility = accessibility.where(~has_desc, accessibility + " – " + wheelchair_desc)

df_clean["accessibility_features"] = accessibility

df_clean.head()

,id,clinic_name,addr_street,addr_housenumber,addr_postcode,addr_city,district,district_id,neighborhood,neighborhood_id,...,contact_phone_raw,email_raw,contact_email_raw,website_raw,contact_website_raw,phone_main,email_main,website_main,contact_info,accessibility_features
0,268917040,Tierarztpraxis am Urban,Baerwaldstraße,69,10961.0,Berlin,Friedrichshain-Kreuzberg,11002002,Kreuzberg,0202,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,no
1,299795048,Dr. med. vet. Elke Hartwig,Straße 48,67,13125.0,Berlin,Pankow,11003003,Karow,0305,...,<NA>,<NA>,<NA>,http://www.tierarztpraxis-hartwig.de/,<NA>,+49 30 9437820,<NA>,http://www.tierarztpraxis-hartwig.de/,phone: +49 30 9437820; web: http://www.tierarz...,limited
2,347294456,Tierarztpraxis Dr. Bernhard Sörensen,Königsberger Straße,36,12207.0,Berlin,Steglitz-Zehlendorf,11006006,Lichterfelde,0602,...,<NA>,<NA>,<NA>,https://www.tierarztpraxis-soerensen.de/,<NA>,+49 30 7738321,<NA>,https://www.tierarztpraxis-soerensen.de/,phone: +49 30 7738321; web: https://www.tierar...,yes
3,394867279,Tierarztpraxis Jeanette Koepsel,<NA>,<NA>,<NA>,<NA>,Spandau,11005005,Siemensstadt,0503,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
4,411550894,Kleintierarztpraxis Berlin Kaulsdorf,Planitzstraße,19,12621.0,Berlin,Marzahn-Hellersdorf,11010010,Kaulsdorf,1003,...,<NA>,info@tierarzt-kaulsdorf.de,<NA>,https://www.tierarzt-kaulsdorf.de/,<NA>,+49 30 53018585,info@tierarzt-kaulsdorf.de,https://www.tierarzt-kaulsdorf.de/,phone: +49 30 53018585; email: info@tierarzt-k...,<NA>


## 4. Optional: reverse geocoding for missing addresses (Nominatim)

For clinics with missing address fields but valid coordinates, we optionally
use Nominatim to backfill `addr_street`, `addr_housenumber`, `addr_postcode`
and `addr_city`. This step is limited to a small number of rows to respect
rate limits and can be disabled if needed.

In [5]:
from geopy.geocoders import Nominatim
from time import sleep

# Initialise Nominatim geolocator
geolocator = Nominatim(user_agent="berlin_vet_clinics_reverse_geocoder")

def reverse_geocode_address(lat, lon):
    """
    Reverse geocode to retrieve address fields from coordinates.
    Returns (street, housenumber, postcode, city) or (None, ...).
    """
    try:
        location = geolocator.reverse(
            (lat, lon),
            exactly_one=True,
            language="de",
            timeout=10,
        )
        if not location:
            return None, None, None, None

        addr = location.raw.get("address", {})
        street = (
            addr.get("road")
            or addr.get("pedestrian")
            or addr.get("footway")
            or addr.get("residential")
        )
        housenumber = addr.get("house_number")
        postcode = addr.get("postcode")
        city = (
            addr.get("city")
            or addr.get("town")
            or addr.get("village")
            or addr.get("suburb")
        )
        return street, housenumber, postcode, city
    except Exception:
        return None, None, None, None

# Select candidates with missing address but valid coordinates
mask_missing_addr = (
    df_clean["addr_street"].isna()
    & df_clean["addr_postcode"].isna()
    & df_clean["addr_city"].isna()
    & df_clean["latitude"].notna()
    & df_clean["longitude"].notna()
)

print("Rows with missing address and valid coords:", mask_missing_addr.sum())

# To control runtime and API usage, you can change this limit
MAX_REVERSE_GEOCODES = 50

for idx, row in df_clean.loc[mask_missing_addr].head(MAX_REVERSE_GEOCODES).iterrows():
    street, hn, pc, city = reverse_geocode_address(row["latitude"], row["longitude"])

    if street and pd.isna(df_clean.at[idx, "addr_street"]):
        df_clean.at[idx, "addr_street"] = street
    if hn and pd.isna(df_clean.at[idx, "addr_housenumber"]):
        df_clean.at[idx, "addr_housenumber"] = hn
    if pc and pd.isna(df_clean.at[idx, "addr_postcode"]):
        df_clean.at[idx, "addr_postcode"] = pc
    if city and pd.isna(df_clean.at[idx, "addr_city"]):
        df_clean.at[idx, "addr_city"] = city

    # Be polite with the public Nominatim service
    sleep(1)

print("Reverse geocoding step finished (limited to", MAX_REVERSE_GEOCODES, "rows).")

Rows with missing address and valid coords: 48
Reverse geocoding step finished (limited to 50 rows).


## 5. Derived and provenance columns

Now that addresses have been backfilled where possible, we:
- Build the final `full_address` (and `address` alias) from address components.
- Add a descriptive `data_source`.
- Carry over `source_osm_id`.
- Add a quality flag `has_minimum_info`.

In [6]:
# 5.1 Full address and address alias
df_addr_tmp = df_clean[["addr_street", "addr_housenumber", "addr_postcode", "addr_city"]]
df_clean["full_address"] = df_addr_tmp.apply(build_full_address, axis=1).astype("string")
df_clean["address"] = df_clean["full_address"]

# 5.2 Data source description
df_clean["data_source"] = (
    "OSM amenity=veterinary, Berlin, fetched via OSMNX latest snapshot"
)

# 5.3 Provenance: keep OSM source id
df_clean["source_osm_id"] = to_str(df["source_osm_id"])

# 5.4 Quality flag: minimum info to be considered usable
df_clean["has_minimum_info"] = (
    df_clean["clinic_name"].notna()
    | df_clean["full_address"].notna()
    | df_clean["phone_main"].notna()
    | df_clean["website_main"].notna()
)

df_clean[["id", "clinic_name", "address", "has_minimum_info"]].head()

,id,clinic_name,address,has_minimum_info
0,268917040,Tierarztpraxis am Urban,"Baerwaldstraße 69, 10961.0 Berlin",True
1,299795048,Dr. med. vet. Elke Hartwig,"Straße 48 67, 13125.0 Berlin",True
2,347294456,Tierarztpraxis Dr. Bernhard Sörensen,"Königsberger Straße 36, 12207.0 Berlin",True
3,394867279,Tierarztpraxis Jeanette Koepsel,"Wernerwerkdamm 27, 13629 Berlin",True
4,411550894,Kleintierarztpraxis Berlin Kaulsdorf,"Planitzstraße 19, 12621.0 Berlin",True


## 6. Normalise NULLs vs empty strings

We ensure key text fields use NULL (`pd.NA`) for missing values instead of
empty strings, so SQL filtering and aggregations remain consistent.

In [7]:
cols_to_null_empty = [
    "clinic_name",
    "addr_street",
    "addr_housenumber",
    "addr_postcode",
    "addr_city",
    "full_address",
    "address",
    "district",
    "district_id",
    "neighborhood",
    "neighborhood_id",
    "lor_id",
    "services_offered",
    "operating_hours",
    "operating_days",
    "phone_main",
    "email_main",
    "website_main",
    "contact_info",
    "accessibility_features",
    "data_source",
]

for col in cols_to_null_empty:
    if col in df_clean.columns:
        df_clean[col] = (
            df_clean[col]
            .astype("string")
            .replace(r"^\s*$", pd.NA, regex=True)
        )

df_clean.head()

,id,clinic_name,addr_street,addr_housenumber,addr_postcode,addr_city,district,district_id,neighborhood,neighborhood_id,...,phone_main,email_main,website_main,contact_info,accessibility_features,full_address,address,data_source,source_osm_id,has_minimum_info
0,268917040,Tierarztpraxis am Urban,Baerwaldstraße,69,10961.0,Berlin,Friedrichshain-Kreuzberg,11002002,Kreuzberg,0202,...,<NA>,<NA>,<NA>,<NA>,no,"Baerwaldstraße 69, 10961.0 Berlin","Baerwaldstraße 69, 10961.0 Berlin","OSM amenity=veterinary, Berlin, fetched via OS...",node/268917040,True
1,299795048,Dr. med. vet. Elke Hartwig,Straße 48,67,13125.0,Berlin,Pankow,11003003,Karow,0305,...,+49 30 9437820,<NA>,http://www.tierarztpraxis-hartwig.de/,phone: +49 30 9437820; web: http://www.tierarz...,limited,"Straße 48 67, 13125.0 Berlin","Straße 48 67, 13125.0 Berlin","OSM amenity=veterinary, Berlin, fetched via OS...",node/299795048,True
2,347294456,Tierarztpraxis Dr. Bernhard Sörensen,Königsberger Straße,36,12207.0,Berlin,Steglitz-Zehlendorf,11006006,Lichterfelde,0602,...,+49 30 7738321,<NA>,https://www.tierarztpraxis-soerensen.de/,phone: +49 30 7738321; web: https://www.tierar...,yes,"Königsberger Straße 36, 12207.0 Berlin","Königsberger Straße 36, 12207.0 Berlin","OSM amenity=veterinary, Berlin, fetched via OS...",node/347294456,True
3,394867279,Tierarztpraxis Jeanette Koepsel,Wernerwerkdamm,27,13629,Berlin,Spandau,11005005,Siemensstadt,0503,...,<NA>,<NA>,<NA>,<NA>,<NA>,"Wernerwerkdamm 27, 13629 Berlin","Wernerwerkdamm 27, 13629 Berlin","OSM amenity=veterinary, Berlin, fetched via OS...",node/394867279,True
4,411550894,Kleintierarztpraxis Berlin Kaulsdorf,Planitzstraße,19,12621.0,Berlin,Marzahn-Hellersdorf,11010010,Kaulsdorf,1003,...,+49 30 53018585,info@tierarzt-kaulsdorf.de,https://www.tierarzt-kaulsdorf.de/,phone: +49 30 53018585; email: info@tierarzt-k...,<NA>,"Planitzstraße 19, 12621.0 Berlin","Planitzstraße 19, 12621.0 Berlin","OSM amenity=veterinary, Berlin, fetched via OS...",node/411550894,True


## 7. Data quality checks

Quick checks on nulls and coverage of key fields.

In [8]:
print("Null counts per column:")
print(df_clean.isna().sum())

print("\nEmpty or NULL clinic_name rows:", df_clean["clinic_name"].isna().sum())
print("NULL district rows:", df_clean["district"].isna().sum())
print("NULL neighborhood rows:", df_clean["neighborhood"].isna().sum())
print("NULL latitude rows:", df_clean["latitude"].isna().sum())
print("NULL longitude rows:", df_clean["longitude"].isna().sum())

print("\nRecord quality (has_minimum_info value counts):")
print(df_clean["has_minimum_info"].value_counts(dropna=False))

Null counts per column:
id                          0
clinic_name                 4
addr_street                 0
addr_housenumber           37
addr_postcode              12
addr_city                  14
district                    0
district_id                 0
neighborhood                0
neighborhood_id             0
lor_id                      0
latitude                    0
longitude                   0
geometry                    0
services_offered          173
operating_hours            50
operating_days             56
phone_raw                 103
contact_phone_raw         150
email_raw                 157
contact_email_raw         163
website_raw                89
contact_website_raw       151
phone_main                 78
email_main                145
website_main               66
contact_info               53
accessibility_features     93
full_address                0
address                     0
data_source                 0
source_osm_id               0
has_minimum_info

## 8. Export v1 rich vet clinics table

We persist the full cleaned table (including helper columns) for internal use
and future enrichment.

In [9]:
from pathlib import Path

output_v1_csv = Path("cache/vet_clinics_berlin_clean_latest_v1.csv")
output_v1_csv.parent.mkdir(parents=True, exist_ok=True)

df_clean.to_csv(output_v1_csv, index=False)

print("Exported cleaned v1 file to:")
print(" -", output_v1_csv)
print("Shape:", df_clean.shape)

Exported cleaned v1 file to:
 - cache/vet_clinics_berlin_clean_latest_v1.csv
Shape: (175, 33)


## 9. Export reduced vet clinics table for unified POI layer

For the unified POI table and the app/backend we provide a reduced schema
focusing on the most important fields.

We intentionally **exclude**:
- `services_offered` (currently sparse, but still available in `df_clean`)
- `operating_days` (information already contained in `operating_hours`)
and keep them only in the rich v1 table.

In [10]:
# Define reduced export schema
export_cols = [
    "id",
    "clinic_name",
    "address",
    "district",
    "district_id",
    "neighborhood",
    "neighborhood_id",
    "lor_id",
    "latitude",
    "longitude",
    "geometry",
    "operating_hours",
    "contact_info",
    "accessibility_features",
    "data_source",
    "source_osm_id",
    "has_minimum_info",
]

df_export = df_clean[export_cols].copy()

print("Export table columns:")
print(df_export.columns.tolist())
print("Export table shape:", df_export.shape)

export_path = Path("cache/vet_clinics_berlin_export_latest_v1.csv")
export_path.parent.mkdir(parents=True, exist_ok=True)
df_export.to_csv(export_path, index=False)

print("Exported reduced export table to:")
print(" -", export_path)

Export table columns:
['id', 'clinic_name', 'address', 'district', 'district_id', 'neighborhood', 'neighborhood_id', 'lor_id', 'latitude', 'longitude', 'geometry', 'operating_hours', 'contact_info', 'accessibility_features', 'data_source', 'source_osm_id', 'has_minimum_info']
Export table shape: (175, 17)
Exported reduced export table to:
 - cache/vet_clinics_berlin_export_latest_v1.csv
